<a href="https://colab.research.google.com/github/Nourmohamed904/Diabetes_RAG_Hackathon/blob/main/Day2_Mariam2_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetes RAG – Ingestion, Chunking, Embeddings & Retrieval

Loads the two diabetes guideline PDFs from GitHub, cleans and organizes them by section, creates page-aware chunks, stores embeddings in Chroma, and tests retrieval.

## 1. Download the PDFs from GitHub

In [1]:
!git clone https://github.com/Nourmohamed904/Diabetes_RAG_Hackathon.git

fatal: destination path 'Diabetes_RAG_Hackathon' already exists and is not an empty directory.


In [2]:
import os

PDF_FOLDER = "/content/Diabetes_RAG_Hackathon/data"

if not os.path.isdir(PDF_FOLDER):
    raise FileNotFoundError(
        f"PDF folder not found: {PDF_FOLDER}\n"
        "Check the repository structure or update PDF_FOLDER."
    )

PDF_PATHS = sorted(
    os.path.join(PDF_FOLDER, file)
    for file in os.listdir(PDF_FOLDER)
    if file.lower().endswith(".pdf")
)

print("Number of PDFs:", len(PDF_PATHS))
for pdf in PDF_PATHS:
    print("-", os.path.basename(pdf))

Number of PDFs: 2
- Type-1 diabetes.pdf
- Type-2 diabetes.pdf


## 2. Install and import the required libraries

In [3]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb fastembed pypdf

In [4]:
import os
import re
import shutil
from collections import Counter

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

/tmp/ipykernel_3497/1583142254.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. Load the PDFs and add metadata

In [5]:
all_pages = []

for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    document_name = os.path.basename(pdf_path)

    for page in pages:
        page.metadata["document_name"] = document_name
        page.metadata["page_number"] = page.metadata.get("page", 0) + 1

    all_pages.extend(pages)

print("Total PDFs:", len(PDF_PATHS))
print("Total pages:", len(all_pages))

Total PDFs: 2
Total pages: 195


## 4. Clean the extracted text

In [6]:
def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for page in all_pages:
    page.page_content = clean_text(page.page_content)

print("Cleaning completed.")

Cleaning completed.


### 4b. Remove repeated boilerplate (copyright footer / page-of-64 lines)

Every page repeats the same NICE copyright notice and a "Page X of NN" footer, and often the guideline title as a running header. This text carries no medical meaning, but it gets embedded into every chunk that touches a page boundary, diluting the chunk's embedding and pushing genuinely relevant chunks further down the similarity ranking. Stripping it before chunking keeps each chunk focused on actual guideline content.

In [7]:
# ==========================================
# 5. Remove repeated boilerplate
# ==========================================

BOILERPLATE_PATTERNS = [
    # NICE copyright/footer
    r"©\s*NICE\s*\d{4}\.\s*All rights reserved\.\s*"
    r"(?:https?://\S+\s*)?"
    r"(?:See\s+notice-of-rights\s*\)?\.?)?",

    # Page X of Y
    r"Page\s+\d+\s+of\s+\d+",

    # Running headers
    r"Type\s+1\s+diabetes\s+in\s+adults:\s*"
    r"diagnosis\s+and\s+management\s*\(NG17\)",

    r"Type\s+2\s+diabetes\s+in\s+adults:\s*"
    r"management\s*\(NG28\)",
]


# Compile each pattern separately.
# This is safer than using one large DOTALL regex.
_BOILERPLATE_REGEXES = [
    re.compile(pattern, flags=re.IGNORECASE)
    for pattern in BOILERPLATE_PATTERNS
]


def remove_boilerplate(text):
    """
    Remove repeated PDF headers/footers without accidentally
    deleting large parts of the medical content.
    """

    for pattern in _BOILERPLATE_REGEXES:
        text = pattern.sub(" ", text)

    # Clean spaces left after removal
    text = re.sub(r"[ \t]+", " ", text)

    # Clean excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ------------------------------------------
# Test before / after
# ------------------------------------------

sample_index = min(16, len(all_pages) - 1)

sample_before = all_pages[sample_index].page_content

sample_after = remove_boilerplate(sample_before)

print("===== BEFORE boilerplate removal =====")
print(sample_before[-500:])

print("\n===== AFTER boilerplate removal =====")
print(sample_after[-500:])


# ------------------------------------------
# Apply to all pages
# ------------------------------------------

for page in all_pages:
    page.page_content = remove_boilerplate(page.page_content)

print(
    "\nBoilerplate removal applied to all",
    len(all_pages),
    "pages"
)

===== BEFORE boilerplate removal =====
able at consultations. Follow the principles on 
communication in NICE's guideline on patient experience in adult NHS services. 
[2015] 
1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or 
abnormal haemoglobin type, estimate trends in blood glucose control using 1 of 
Type 1 diabetes in adults: diagnosis and management (NG17)
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).
Page 17 of
64

===== AFTER boilerplate removal =====
with type 1 diabetes their HbA1c results after each measurement and 
have their most recent result available at consultations. Follow the principles on 
communication in NICE's guideline on patient experience in adult NHS services. 
[2015] 
1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or 
abnormal haemoglobin type, estimate trends in blood glucose control using 1 of 
 
 Subject to Not

In [8]:
print("Document:", all_pages[0].metadata["document_name"])
print("Page:", all_pages[0].metadata["page_number"])
print("\nSample text:\n")
print(all_pages[0].page_content[:1500])

Document: Type-1 diabetes.pdf
Page: 1

Sample text:

Type 1 diabetes in adults: 
diagnosis and management 
NICE guideline 
Published: 26 August 2015 
Last updated: 17 August 2022 
www.nice.org.uk/guidance/ng17 
 Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).


## 5. Detect and clean section headings

In [9]:
def is_section_heading(line):
    line = line.strip()
    if not line:
        return False

    # Main sections such as 1.1, 1.2, 1.10
    pattern = r"^\d+\.\d+\s+.+"
    return bool(re.match(pattern, line))


def clean_section_heading(line):
    line = line.strip()

    # Example:
    # 1.1 Diagnosis and early care plan ............ 6
    # -> 1.1 Diagnosis and early care plan
    line = re.sub(r"\s*\.{3,}\s*\d+\s*$", "", line)

    return line.strip()

In [10]:
test_lines = [
    "1.1 Diagnosis and early care plan",
    "1.2 Support and individualised care",
    "1.10 Ketone monitoring and managing diabetic ketoacidosis",
    "1.1.1 Make an initial diagnosis of type 1 diabetes",
    "People with diabetes should..."
]

for line in test_lines:
    print(is_section_heading(line), "→", line)

True → 1.1 Diagnosis and early care plan
True → 1.2 Support and individualised care
True → 1.10 Ketone monitoring and managing diabetic ketoacidosis
False → 1.1.1 Make an initial diagnosis of type 1 diabetes
False → People with diabetes should...


## 6. Group pages by section

In [11]:
def get_first_content_page(pages):
    """
    Detect the first page containing a real section heading.
    """

    for page in pages:
        for line in page.page_content.splitlines():
            line = line.strip()

            if is_section_heading(line):
                return page.metadata["page_number"]

    return 1


def group_pages_by_section(pages):

    grouped_sections = []

    # Group pages by PDF/document
    pages_by_document = {}

    for page in pages:
        document_name = page.metadata["document_name"]

        pages_by_document.setdefault(
            document_name, []
        ).append(page)

    # Process each PDF independently
    for document_name, document_pages in pages_by_document.items():

        current_section = None
        current_pages = []

        # Detect first content page automatically
        first_content_page = get_first_content_page(
            document_pages
        )

        print(
            f"{document_name} -> "
            f"first content page: {first_content_page}"
        )

        for page in document_pages:

            page_number = page.metadata["page_number"]

            if page_number < first_content_page:
                continue

            for raw_line in page.page_content.splitlines():

                line = raw_line.strip()

                if not line:
                    continue

                # New section
                if is_section_heading(line):

                    if current_section is not None:
                        grouped_sections.append({
                            "document_name": document_name,
                            "section": current_section,
                            "pages": current_pages.copy()
                        })

                    current_section = clean_section_heading(line)

                    current_pages = [{
                        "page_number": page_number,
                        "text": ""
                    }]

                # Normal content
                elif current_section is not None:

                    if (
                        not current_pages
                        or current_pages[-1]["page_number"] != page_number
                    ):
                        current_pages.append({
                            "page_number": page_number,
                            "text": line
                        })

                    else:
                        if current_pages[-1]["text"]:
                            current_pages[-1]["text"] += "\n" + line
                        else:
                            current_pages[-1]["text"] = line

        # Save final section
        if current_section is not None:
            grouped_sections.append({
                "document_name": document_name,
                "section": current_section,
                "pages": current_pages.copy()
            })

    return grouped_sections


# Run
grouped_sections = group_pages_by_section(all_pages)

print("Number of grouped sections:", len(grouped_sections))

if not grouped_sections:
    raise ValueError(
        "No sections were detected. "
        "Check section-heading detection and PDF extraction."
    )

first_section = grouped_sections[0]

print("\n===== FIRST SECTION =====")
print("Document:", first_section["document_name"])
print("Section:", first_section["section"])
print("Number of pages:", len(first_section["pages"]))

print(
    "Pages:",
    [p["page_number"] for p in first_section["pages"]]
)

print("\nFirst page text:")
print(first_section["pages"][0]["text"][:1000])

Type-1 diabetes.pdf -> first content page: 3
Type-2 diabetes.pdf -> first content page: 3
Number of grouped sections: 118

===== FIRST SECTION =====
Document: Type-1 diabetes.pdf
Section: 1.1 Diagnosis and early care plan
Number of pages: 1
Pages: [3]

First page text:



**Chunk size note:** originally 850/150. Lowered to **300/60** after diagnosing Q5 ("insulin plan" question): at 850 chars, recommendation 1.7.1 (insulin regimen) was sharing a chunk with the unrelated "Levemir discontinuation" notice right after it, diluting the chunk's embedding enough that it dropped out of the top-6 results. At 300 chars, 1.7.1 gets an isolated chunk. Verified this doesn't break the other working recommendations (1.6.6 HbA1c target, 1.13.1 metformin first-line) — both still land cleanly in their own chunks at this size.

## 7. Create page-aware chunks

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def create_all_chunks(grouped_sections, splitter):
    all_chunks = []
    chunk_counter = 1

    for section in grouped_sections:
        section_text = ""
        page_boundaries = []

        for page in section["pages"]:
            start_position = len(section_text)
            section_text += page["text"] + "\n"
            end_position = len(section_text)

            page_boundaries.append({
                "page_number": page["page_number"],
                "start": start_position,
                "end": end_position,
            })

        if not page_boundaries:
            continue

        section_chunks = splitter.split_text(section_text)
        search_start = 0

        for chunk_text in section_chunks:
            chunk_start = section_text.find(chunk_text, search_start)
            if chunk_start == -1:
                chunk_start = search_start

            chunk_end = chunk_start + len(chunk_text)

            chunk_pages = [
                boundary["page_number"]
                for boundary in page_boundaries
                if boundary["end"] > chunk_start
                and boundary["start"] < chunk_end
            ]

            if not chunk_pages:
                chunk_pages = [page_boundaries[0]["page_number"]]

            chunk = Document(
                page_content=chunk_text,
                metadata={
                    "document_name": section["document_name"],
                    "section": section["section"],
                    "page_number": chunk_pages[0],
                    "page_numbers": chunk_pages,
                    "chunk_id": f"chunk_{chunk_counter:04d}",
                },
            )

            all_chunks.append(chunk)
            chunk_counter += 1
            search_start = chunk_start + 1

    return all_chunks

In [13]:
chunks = create_all_chunks(grouped_sections, splitter)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3], 1):
    print("=" * 80)
    print(f"CHUNK {i}")
    print("Text:", chunk.page_content[:500])
    print("Metadata:", chunk.metadata)

Total chunks: 1170
CHUNK 1
Text: Terms used in this guideline ................................................................................................................. 48
Recommendations for research .................................................................................................49
Metadata: {'document_name': 'Type-1 diabetes.pdf', 'section': '1.14 Managing complications', 'page_number': 3, 'page_numbers': [3], 'chunk_id': 'chunk_0001'}
CHUNK 2
Text: 1 Clinical features for distinguishing between type 1 diabetes and other types of diabetes ........ 49
2 The use of C-peptide in diagnosing diabetes ................................................................................. 49
Metadata: {'document_name': 'Type-1 diabetes.pdf', 'section': '1.14 Managing complications', 'page_number': 3, 'page_numbers': [3], 'chunk_id': 'chunk_0002'}
CHUNK 3
Text: 3 Use of routinely collected real-world data to examine the effectiveness and cost
effectiveness of continuous glu

## 8. Validate chunk metadata

In [14]:
required_fields = [
    "document_name",
    "section",
    "page_number",
    "page_numbers",
    "chunk_id",
]

for i, chunk in enumerate(chunks):
    for field in required_fields:
        assert field in chunk.metadata, f"Missing '{field}' in chunk {i}"

chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate chunk IDs found"

print("Metadata validation passed.")
print("Unique chunks:", len(chunks))

document_counts = Counter(
    chunk.metadata["document_name"] for chunk in chunks
)

print("\nChunks per document:")
for document, count in document_counts.items():
    print("-", document, ":", count)

Metadata validation passed.
Unique chunks: 1170

Chunks per document:
- Type-1 diabetes.pdf : 393
- Type-2 diabetes.pdf : 777


## 9. Generate embeddings and create the Chroma vector store

In [15]:
import os
import shutil
from tqdm.auto import tqdm
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = FastEmbedEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

cleaned_chunks = []
for doc in chunks:
    clean_meta = {}
    for key, value in doc.metadata.items():
        if value is None:
            clean_meta[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean_meta[key] = value
        else:
            clean_meta[key] = str(value)

    doc.metadata = clean_meta
    cleaned_chunks.append(doc)

persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vector_db = Chroma(
    collection_name="diabetes_educational_rag",
    embedding_function=embedding_model,
    persist_directory=persist_directory,
)

batch_size = 64
for i in tqdm(range(0, len(cleaned_chunks), batch_size), desc="Indexing to Chroma"):
    batch = cleaned_chunks[i:i + batch_size]
    vector_db.add_documents(batch)

stored_count = vector_db._collection.count()

print("Input chunks :", len(chunks))
print("Stored vectors:", stored_count)

assert stored_count == len(chunks), "Chroma count does not match input chunks."
print("Chroma indexing completed successfully.")

/tmp/ipykernel_3497/4201356587.py:30: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


Indexing to Chroma:   0%|          | 0/19 [00:00<?, ?it/s]

Input chunks : 1170
Stored vectors: 1170
Chroma indexing completed successfully.


## 10. Retrieval test

In [16]:
def _extract_pages(chunk_pages, fallback_page):
    if isinstance(chunk_pages, list):
        return [
            int(p)
            for p in chunk_pages
            if str(p).strip().lstrip("-").isdigit()
        ]

    if isinstance(chunk_pages, str):
        found = re.findall(r"\d+", chunk_pages)
        if found:
            return [int(p) for p in found]

    return [fallback_page] if fallback_page is not None else []


def retrieve_with_similarity(question, k=4):
    return vector_db.similarity_search_with_relevance_scores(question, k=k)


def print_retrieval_results(question, k=4):
    results = retrieve_with_similarity(question, k=k)

    print(f"QUESTION: {question}\n")

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"Rank {rank}")
        print("Document :", doc.metadata.get("document_name"))
        print("Page     :", doc.metadata.get("page_number"))
        print("Pages    :", doc.metadata.get("page_numbers"))
        print("Section  :", doc.metadata.get("section"))
        print("Chunk ID :", doc.metadata.get("chunk_id"))
        print("Score    :", round(score, 4))
        print("Text     :", doc.page_content[:500].replace("\n", " "), "...")
        print()

    return results

In [17]:
results = print_retrieval_results(
    "What are the diagnostic criteria for diabetes?",
    k=4,
)

QUESTION: What are the diagnostic criteria for diabetes?

Rank 1
Document : Type-1 diabetes.pdf
Page     : 6
Pages    : [6]
Section  : 1.1 Diagnosis and early care plan
Chunk ID : chunk_0015
Score    : 0.6633
Text     : Initial diagnosis 1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes typically (but not always) have 1 or more of: • ketosis • rapid weight loss • age of onset under 50 years ...

Rank 2
Document : Type-1 diabetes.pdf
Page     : 7
Pages    : [7]
Section  : 1.1 Diagnosis and early care plan
Chunk ID : chunk_0017
Score    : 0.6068
Text     : 2022] 1.1.2 Do not use age or BMI alone to exclude or diagnose type 1 diabetes in adults. [2022] 1.1.3 Take into consideration the possibility of other diabetes subtypes and revisit the diagnosis at subsequent clinical reviews. Carry out further investigations if there ...

Rank 3
Document : Type-1 diabetes.pdf
Page     : 5
Page

**Note:** the test questions are now hardcoded directly in this notebook instead of parsed
from `RAG_Test_Questions.pdf`. Parsing a PDF table with regex is fragile — it silently breaks
(and produces `0/0` results) whenever the PDF's layout changes. Editing questions here is
also easier: just edit the list below.


In [18]:

supported_questions = [
    {"number": 1,  "question": "What should my HbA1c number be if I have type 1 diabetes?", "expected_source": "type-1.pdf", "expected_page": 18, "expected_keywords": ["48", "6.5"]},
    {"number": 2,  "question": "How often should I get my HbA1c checked?", "expected_source": "type-1.pdf", "expected_page": 17, "expected_keywords": ["3 to 6 months"]},
    {"number": 3,  "question": "What are the signs that someone might have type 1 diabetes?", "expected_source": "type-1.pdf", "expected_page": 6, "expected_keywords": ["ketosis", "weight loss"]},
    {"number": 4,  "question": "Can a doctor tell I have type 1 diabetes just from my age or weight?", "expected_source": "type-1.pdf", "expected_page": 7, "expected_keywords": ["BMI"]},
    {"number": 5,  "question": "What kind of insulin plan do people with type 1 diabetes usually start with?", "expected_source": "type-1.pdf", "expected_page": 24, "expected_keywords": ["basal", "bolus"]},
    {"number": 6,  "question": "When do I need to check my blood sugar more than 10 times a day?", "expected_source": "type-1.pdf", "expected_page": 23, "expected_keywords": ["10 times"]},
    {"number": 7,  "question": "What should be done if someone with diabetes passes out from low blood sugar?", "expected_source": "type-1.pdf", "expected_page": 31, "expected_keywords": ["glucagon"]},
    {"number": 8,  "question": "What blood sugar level should I aim for before an operation?", "expected_source": "type-1.pdf", "expected_page": 38, "expected_keywords": ["5 to 8"]},
    {"number": 9,  "question": "How often should someone with type 2 diabetes get their HbA1c checked?", "expected_source": "type-2.pdf", "expected_page": 12, "expected_keywords": ["HbA1c"]},
    {"number": 10, "question": "Do I need to check my blood sugar every day if I have type 2 diabetes?", "expected_source": "type-2.pdf", "expected_page": 14, "expected_keywords": ["self-monitoring"]},
    {"number": 11, "question": "What's usually the first medicine given for type 2 diabetes?", "expected_source": "type-2.pdf", "expected_page": 32, "expected_keywords": ["metformin"]},
    {"number": 12, "question": "What treatment is given if someone has type 2 diabetes and a heart problem?", "expected_source": "type-2.pdf", "expected_page": 40, "expected_keywords": ["heart failure"]},
    {"number": 13, "question": "What treatment options are there for someone with type 2 diabetes who is overweight?", "expected_source": "type-2.pdf", "expected_page": 58, "expected_keywords": ["obesity"]},
    {"number": 14, "question": "What should be checked before starting SGLT-2 medicine?", "expected_source": "type-2.pdf", "expected_page": 70, "expected_keywords": ["SGLT-2"]},
    {"number": 15, "question": "Should I take aspirin to protect my heart if I have type 2 diabetes?", "expected_source": "type-2.pdf", "expected_page": 119, "expected_keywords": ["antiplatelet", "aspirin"]},
]

unsupported_questions = [
    {"number": 1, "question": "How much do diabetes medicines cost in Egypt?", "reason": "No pricing or local market info in either PDF"},
    {"number": 2, "question": "Is there a cure for diabetes now?", "reason": "Not mentioned in either guideline"},
    {"number": 3, "question": "Can people with type 1 diabetes take weight-loss injections like Ozempic?", "reason": "type-1.pdf does not cover this type of medicine at all"},
    {"number": 4, "question": "Will type 1 diabetes affect my chances of having children?", "reason": "type-1.pdf just points to a separate pregnancy guideline, no real answer given"},
    {"number": 5, "question": "Which brand of blood sugar monitor is the best one to buy?", "reason": "The guidelines don't recommend specific brands or products"},
]

print("Supported questions  :", len(supported_questions))
print("Unsupported questions:", len(unsupported_questions))

Supported questions  : 15
Unsupported questions: 5


In [19]:
def check_match(
    results,
    expected_source,
    expected_page,
    expected_keywords=None,
    page_tolerance=3
):
    """
    Checks whether at least one retrieved chunk:
      1. Comes from the expected guideline.
      2. Is within the expected page tolerance.
      3. Contains ALL expected keywords.

    Returns:
        page_match
        keyword_match
        match
    """

    expected_source = (expected_source or "").lower()

    # Identify expected guideline
    if "type-1" in expected_source:
        expected_tag = "type-1"
    elif "type-2" in expected_source:
        expected_tag = "type-2"
    else:
        expected_tag = expected_source

    expected_keywords = [
        kw.lower().strip()
        for kw in (expected_keywords or [])
        if kw and kw.strip()
    ]

    page_match = False
    keyword_match = False
    full_match = False

    for doc, _score in results:

        # -----------------------------
        # Document / source
        # -----------------------------
        doc_name = doc.metadata.get(
            "document_name",
            ""
        ).lower()

        source_match = expected_tag in doc_name

        if not source_match:
            continue

        # -----------------------------
        # Pages
        # -----------------------------
        chunk_pages = _extract_pages(
            doc.metadata.get("page_numbers"),
            doc.metadata.get("page_number")
        )

        current_page_match = any(
            abs(page - expected_page) <= page_tolerance
            for page in chunk_pages
        )

        if not current_page_match:
            continue

        # We found the expected source + page
        page_match = True

        # -----------------------------
        # Keywords
        # -----------------------------
        chunk_text = doc.page_content.lower()

        if expected_keywords:

            current_keyword_match = all(
                keyword in chunk_text
                for keyword in expected_keywords
            )

        else:
            current_keyword_match = True

        if current_keyword_match:
            keyword_match = True
            full_match = True

            # No need to check remaining chunks
            break

    return {
        "page_match": page_match,
        "keyword_match": keyword_match,
        "match": full_match,
    }

In [20]:
print("===== UNSUPPORTED QUESTIONS =====\n")

for item in unsupported_questions:
    results = print_retrieval_results(item["question"], k=4)
    top_score = results[0][1] if results else 0

    print("Why unsupported:", item["reason"])
    print("Top-1 similarity:", round(top_score, 4))
    print("=" * 80)

===== UNSUPPORTED QUESTIONS =====

QUESTION: How much do diabetes medicines cost in Egypt?

Rank 1
Document : Type-2 diabetes.pdf
Page     : 54
Pages    : [54]
Section  : 1.16 People with early onset type 2 diabetes
Chunk ID : chunk_0723
Score    : 0.4987
Text     : effective, while adding liraglutide to an SGLT-2 inhibitor and metformin reported an incremental cost-effectiveness ratio (ICER) approaching £20,000 per quality-adjusted life year (QALY) gained. Tirzepatide was not analysed for this population. ...

Rank 2
Document : Type-2 diabetes.pdf
Page     : 125
Pages    : [125]
Section  : 1.45 Antiplatelet therapy
Chunk ID : chunk_1131
Score    : 0.4987
Text     : effective, while adding liraglutide to an SGLT-2 inhibitor and metformin reported an incremental cost-effectiveness ratio (ICER) approaching £20,000 per quality-adjusted life year (QALY) gained. Tirzepatide was not analysed for this population. ...

Rank 3
Document : Type-2 diabetes.pdf
Page     : 41
Pages    : [41]
Section

## Final status

In [21]:
print("===== RAG DAY 1 STATUS =====")
print("PDFs loaded       :", len(PDF_PATHS))
print("Pages loaded      :", len(all_pages))
print("Grouped sections  :", len(grouped_sections))
print("Chunks created    :", len(chunks))
print("Vectors stored    :", vector_db._collection.count())
print("Supported tests   :", len(supported_questions))
print("Unsupported tests :", len(unsupported_questions))

===== RAG DAY 1 STATUS =====
PDFs loaded       : 2
Pages loaded      : 195
Grouped sections  : 118
Chunks created    : 1170
Vectors stored    : 1170
Supported tests   : 15
Unsupported tests : 5


## Top-K Experiment

In [22]:
TOP_K_VALUES = [3, 5, 10]

test_questions = {
    1: "What should my HbA1c number be if I have type 1 diabetes?",
    2: "How often should I get my HbA1c checked?",
    3: "What are the signs that someone might have type 1 diabetes?",
    5: "What kind of insulin plan do people with type 1 diabetes usually start with?",
    9: "How often should someone with type 2 diabetes get their HbA1c checked?"
}

In [23]:
for q_id, question in test_questions.items():
    print(q_id, "->", question)

1 -> What should my HbA1c number be if I have type 1 diabetes?
2 -> How often should I get my HbA1c checked?
3 -> What are the signs that someone might have type 1 diabetes?
5 -> What kind of insulin plan do people with type 1 diabetes usually start with?
9 -> How often should someone with type 2 diabetes get their HbA1c checked?


In [24]:
question = test_questions[1]
results = retrieve_with_similarity(question, k=3)

print(f"Question: {question}")
print(f"Number of retrieved chunks: {len(results)}")

Question: What should my HbA1c number be if I have type 1 diabetes?
Number of retrieved chunks: 3


In [25]:
def display_retrieval_results(question, k):
    results = retrieve_with_similarity(question, k=k)

    print("=" * 100)
    print(f"Question: {question}")
    print(f"Top-K: {k}")
    print("=" * 100)

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\nRank: {rank}")
        print(f"Similarity Score: {score:.4f}")
        print(f"Document: {doc.metadata.get('document_name')}")
        print(f"Page: {doc.metadata.get('page_number')}")
        print(f"Section: {doc.metadata.get('section')}")
        print(f"Chunk ID: {doc.metadata.get('chunk_id')}")
        print("\nChunk Text:")
        print(doc.page_content)
        print("-" * 100)

In [26]:
display_retrieval_results(
    test_questions[1],
    k=3)

Question: What should my HbA1c number be if I have type 1 diabetes?
Top-K: 3

Rank: 1
Similarity Score: 0.6732
Document: Type-1 diabetes.pdf
Page: 17
Section: 1.6 Blood glucose management
Chunk ID: chunk_0078

Chunk Text:
HbA1c measurement and targets
Measurement
1.6.1 Measure HbA1c levels every 3 to 6 months in adults with type 1 diabetes. [2015]
1.6.2 Consider measuring HbA1c levels more often in adults with type 1 diabetes if their
blood glucose control is suspected to be changing rapidly; for example, if their
----------------------------------------------------------------------------------------------------

Rank: 2
Similarity Score: 0.6476
Document: Type-1 diabetes.pdf
Page: 23
Section: 1.6 Blood glucose management
Chunk ID: chunk_0113

Chunk Text:
Blood glucose targets
1.6.22 Advise adults with type 1 diabetes to aim for:
• a fasting plasma glucose level of 5 to 7 mmol/litre on waking and
• a plasma glucose level of 4 to 7 mmol/litre before meals at other times of the
day. [201

In [27]:
display_retrieval_results(
    test_questions[1],
    k=5
)

Question: What should my HbA1c number be if I have type 1 diabetes?
Top-K: 5

Rank: 1
Similarity Score: 0.6732
Document: Type-1 diabetes.pdf
Page: 17
Section: 1.6 Blood glucose management
Chunk ID: chunk_0078

Chunk Text:
HbA1c measurement and targets
Measurement
1.6.1 Measure HbA1c levels every 3 to 6 months in adults with type 1 diabetes. [2015]
1.6.2 Consider measuring HbA1c levels more often in adults with type 1 diabetes if their
blood glucose control is suspected to be changing rapidly; for example, if their
----------------------------------------------------------------------------------------------------

Rank: 2
Similarity Score: 0.6476
Document: Type-1 diabetes.pdf
Page: 23
Section: 1.6 Blood glucose management
Chunk ID: chunk_0113

Chunk Text:
Blood glucose targets
1.6.22 Advise adults with type 1 diabetes to aim for:
• a fasting plasma glucose level of 5 to 7 mmol/litre on waking and
• a plasma glucose level of 4 to 7 mmol/litre before meals at other times of the
day. [201

In [28]:
display_retrieval_results(
    test_questions[1],
    k=10
)

Question: What should my HbA1c number be if I have type 1 diabetes?
Top-K: 10

Rank: 1
Similarity Score: 0.6732
Document: Type-1 diabetes.pdf
Page: 17
Section: 1.6 Blood glucose management
Chunk ID: chunk_0078

Chunk Text:
HbA1c measurement and targets
Measurement
1.6.1 Measure HbA1c levels every 3 to 6 months in adults with type 1 diabetes. [2015]
1.6.2 Consider measuring HbA1c levels more often in adults with type 1 diabetes if their
blood glucose control is suspected to be changing rapidly; for example, if their
----------------------------------------------------------------------------------------------------

Rank: 2
Similarity Score: 0.6476
Document: Type-1 diabetes.pdf
Page: 23
Section: 1.6 Blood glucose management
Chunk ID: chunk_0113

Chunk Text:
Blood glucose targets
1.6.22 Advise adults with type 1 diabetes to aim for:
• a fasting plasma glucose level of 5 to 7 mmol/litre on waking and
• a plasma glucose level of 4 to 7 mmol/litre before meals at other times of the
day. [20

In [29]:
TOP_K_VALUES = [3, 5, 10]
all_results = []
for q_id, question in test_questions.items():
    for k in TOP_K_VALUES:
        results = retrieve_with_similarity(question, k=k)
        for rank, (doc, score) in enumerate(results, start=1):

            all_results.append({
                "Query ID": q_id,
                "Question": question,
                "Rank": rank,
                "Chunk ID": doc.metadata.get("chunk_id"),
                "Document": doc.metadata.get("document_name"),
                "Page": doc.metadata.get("page_number"),
                "Section": doc.metadata.get("section"),
                "Similarity Score": score,
                "Chunk Text": doc.page_content,
                "K": k
            })

In [30]:
import pandas as pd
results_df = pd.DataFrame(all_results)
results_df.head()

,Query ID,Question,Rank,Chunk ID,Document,Page,Section,Similarity Score,Chunk Text,K
0,1,What should my HbA1c number be if I have type ...,1,chunk_0078,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.673183,HbA1c measurement and targets\nMeasurement\n1....,3
1,1,What should my HbA1c number be if I have type ...,2,chunk_0113,Type-1 diabetes.pdf,23,1.6 Blood glucose management,0.647586,Blood glucose targets\n1.6.22 Advise adults wi...,3
2,1,What should my HbA1c number be if I have type ...,3,chunk_0085,Type-1 diabetes.pdf,18,1.6 Blood glucose management,0.644524,who reach an HbA1c level of 53 mmol/mol (7%) o...,3
3,1,What should my HbA1c number be if I have type ...,1,chunk_0078,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.673183,HbA1c measurement and targets\nMeasurement\n1....,5
4,1,What should my HbA1c number be if I have type ...,2,chunk_0113,Type-1 diabetes.pdf,23,1.6 Blood glucose management,0.647586,Blood glucose targets\n1.6.22 Advise adults wi...,5


In [31]:
print("Number of rows:", len(results_df))
print("Number of questions:", results_df["Query ID"].nunique())
print("K values:", results_df["K"].unique())

Number of rows: 90
Number of questions: 5
K values: [ 3  5 10]


In [32]:
results_df[
    [
        "Query ID",
        "Rank",
        "Chunk ID",
        "Document",
        "Page",
        "Section",
        "Similarity Score",
        "K"
    ]
].head(15)

,Query ID,Rank,Chunk ID,Document,Page,Section,Similarity Score,K
0,1,1,chunk_0078,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.673183,3
1,1,2,chunk_0113,Type-1 diabetes.pdf,23,1.6 Blood glucose management,0.647586,3
2,1,3,chunk_0085,Type-1 diabetes.pdf,18,1.6 Blood glucose management,0.644524,3
3,1,1,chunk_0078,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.673183,5
4,1,2,chunk_0113,Type-1 diabetes.pdf,23,1.6 Blood glucose management,0.647586,5
5,1,3,chunk_0085,Type-1 diabetes.pdf,18,1.6 Blood glucose management,0.644524,5
6,1,4,chunk_0080,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.639599,5
7,1,5,chunk_0082,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.638873,5
8,1,1,chunk_0078,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.673183,10
9,1,2,chunk_0113,Type-1 diabetes.pdf,23,1.6 Blood glucose management,0.647586,10


In [33]:
results_df.groupby(["Query ID", "K"]).size()

Query ID  K 
1         3      3
          5      5
          10    10
2         3      3
          5      5
          10    10
3         3      3
          5      5
          10    10
5         3      3
          5      5
          10    10
9         3      3
          5      5
          10    10
dtype: int64

In [34]:
results_df["Relevant?"] = ""

In [35]:
labeling_df = (
    results_df
    .sort_values(["Query ID", "Rank"])
    .drop_duplicates(subset=["Query ID", "Chunk ID"])
    .copy()
)

labeling_df["Relevant?"] = ""

In [36]:
print("Unique question-chunk pairs:", len(labeling_df))

Unique question-chunk pairs: 50


In [37]:
labeling_df[
    [
        "Query ID",
        "Question",
        "Rank",
        "Chunk ID",
        "Document",
        "Page",
        "Section",
        "Similarity Score",
        "Relevant?",
        "Chunk Text"
    ]
].head(20)

,Query ID,Question,Rank,Chunk ID,Document,Page,Section,Similarity Score,Relevant?,Chunk Text
0,1,What should my HbA1c number be if I have type ...,1,chunk_0078,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.673183,,HbA1c measurement and targets\nMeasurement\n1....
1,1,What should my HbA1c number be if I have type ...,2,chunk_0113,Type-1 diabetes.pdf,23,1.6 Blood glucose management,0.647586,,Blood glucose targets\n1.6.22 Advise adults wi...
2,1,What should my HbA1c number be if I have type ...,3,chunk_0085,Type-1 diabetes.pdf,18,1.6 Blood glucose management,0.644524,,who reach an HbA1c level of 53 mmol/mol (7%) o...
6,1,What should my HbA1c number be if I have type ...,4,chunk_0080,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.639599,,Clinical Chemistry (IFCC) standardisation. [20...
7,1,What should my HbA1c number be if I have type ...,5,chunk_0082,Type-1 diabetes.pdf,17,1.6 Blood glucose management,0.638873,,conditions#notice-of-rights).\nthe following:\...
13,1,What should my HbA1c number be if I have type ...,6,chunk_0083,Type-1 diabetes.pdf,18,1.6 Blood glucose management,0.638856,,"mol (6.5%) or lower, to minimise the risk of l..."
14,1,What should my HbA1c number be if I have type ...,7,chunk_0015,Type-1 diabetes.pdf,6,1.1 Diagnosis and early care plan,0.633214,,Initial diagnosis\n1.1.1 Make an initial diagn...
15,1,What should my HbA1c number be if I have type ...,8,chunk_0084,Type-1 diabetes.pdf,18,1.6 Blood glucose management,0.617457,,"complications, comorbidities, occupation and h..."
16,1,What should my HbA1c number be if I have type ...,9,chunk_0448,Type-2 diabetes.pdf,12,1.5 HbA1c measurement and targets,0.616883,,Measurement\n1.5.1 Measure HbA1c levels in adu...
17,1,What should my HbA1c number be if I have type ...,10,chunk_0059,Type-1 diabetes.pdf,13,1.4 Dietary management,0.599663,,Carbohydrate counting\n1.4.1 Offer carbohydrat...


In [38]:
results_df["Relevant?"] = ""

labeling_df = (
    results_df
    .sort_values(["Query ID", "Rank"])
    .drop_duplicates(subset=["Query ID", "Chunk ID"])
    .copy()
)

labeling_df["Relevant?"] = ""

print("Unique question-chunk pairs:", len(labeling_df))

Unique question-chunk pairs: 50


In [39]:
q1_labels = {
    "chunk_0078": "NO",
    "chunk_0113": "NO",
    "chunk_0085": "YES",
    "chunk_0080": "NO",
    "chunk_0083": "YES",
    "chunk_0082": "YES",
    "chunk_0015": "NO",
    "chunk_0084": "YES",
    "chunk_0448": "NO",
    "chunk_0059": "NO"
}

In [40]:
for chunk_id, label in q1_labels.items():
    labeling_df.loc[
        (labeling_df["Query ID"] == 1) &
        (labeling_df["Chunk ID"] == chunk_id),
        "Relevant?"
    ] = label

In [41]:
labeling_df[labeling_df["Query ID"] == 1][
    ["Rank", "Chunk ID", "Similarity Score", "Relevant?"]
]

,Rank,Chunk ID,Similarity Score,Relevant?
0,1,chunk_0078,0.673183,NO
1,2,chunk_0113,0.647586,NO
2,3,chunk_0085,0.644524,YES
6,4,chunk_0080,0.639599,NO
7,5,chunk_0082,0.638873,YES
13,6,chunk_0083,0.638856,YES
14,7,chunk_0015,0.633214,NO
15,8,chunk_0084,0.617457,YES
16,9,chunk_0448,0.616883,NO
17,10,chunk_0059,0.599663,NO


In [42]:
q1_labels = {
    "chunk_0078": "NO",
    "chunk_0113": "NO",
    "chunk_0085": "YES",
    "chunk_0080": "NO",
    "chunk_0083": "YES",
    "chunk_0082": "YES",
    "chunk_0015": "NO",
    "chunk_0084": "YES",
    "chunk_0448": "NO",
    "chunk_0059": "NO"
}

for chunk_id, label in q1_labels.items():
    labeling_df.loc[
        (labeling_df["Query ID"] == 1) &
        (labeling_df["Chunk ID"] == chunk_id),
        "Relevant?"
    ] = label

labeling_df[labeling_df["Query ID"] == 1][
    ["Rank", "Chunk ID", "Similarity Score", "Relevant?"]
]

,Rank,Chunk ID,Similarity Score,Relevant?
0,1,chunk_0078,0.673183,NO
1,2,chunk_0113,0.647586,NO
2,3,chunk_0085,0.644524,YES
6,4,chunk_0080,0.639599,NO
7,5,chunk_0082,0.638873,YES
13,6,chunk_0083,0.638856,YES
14,7,chunk_0015,0.633214,NO
15,8,chunk_0084,0.617457,YES
16,9,chunk_0448,0.616883,NO
17,10,chunk_0059,0.599663,NO


In [43]:
q2_labels = {
    "chunk_0078": "YES",
    "chunk_0448": "YES",
    "chunk_0107": "NO",
    "chunk_0475": "NO",
    "chunk_1142": "NO",
    "chunk_0223": "NO",
    "chunk_0110": "NO",
    "chunk_0495": "NO",
    "chunk_0079": "NO",
    "chunk_0106": "NO"
}

for chunk_id, label in q2_labels.items():
    labeling_df.loc[
        (labeling_df["Query ID"] == 2) &
        (labeling_df["Chunk ID"] == chunk_id),
        "Relevant?"
    ] = label

In [44]:
labeling_df[labeling_df["Query ID"] == 2][
    ["Rank", "Chunk ID", "Similarity Score", "Relevant?"]
]

,Rank,Chunk ID,Similarity Score,Relevant?
18,1,chunk_0078,0.774129,YES
19,2,chunk_0448,0.708458,YES
20,3,chunk_0107,0.702986,NO
24,4,chunk_0475,0.663097,NO
25,5,chunk_1142,0.619667,NO
31,6,chunk_0223,0.606000,NO
32,7,chunk_0110,0.604883,NO
33,8,chunk_0495,0.603763,NO
34,9,chunk_0079,0.599271,NO
35,10,chunk_0106,0.596401,NO


In [45]:
q3 = labeling_df[labeling_df["Query ID"] == 3].sort_values("Rank")

for _, row in q3.iterrows():
    print("=" * 100)
    print(f"Rank: {row['Rank']}")
    print(f"Chunk ID: {row['Chunk ID']}")
    print(f"Document: {row['Document']}")
    print(f"Page: {row['Page']}")
    print(f"Section: {row['Section']}")
    print(f"Similarity Score: {row['Similarity Score']:.4f}")
    print("\nChunk Text:")
    print(row["Chunk Text"])
    print()

Rank: 1
Chunk ID: chunk_0015
Document: Type-1 diabetes.pdf
Page: 6
Section: 1.1 Diagnosis and early care plan
Similarity Score: 0.6522

Chunk Text:
Initial diagnosis
1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults
presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes
typically (but not always) have 1 or more of:
• ketosis
• rapid weight loss
• age of onset under 50 years

Rank: 2
Chunk ID: chunk_0070
Document: Type-1 diabetes.pdf
Page: 15
Section: 1.4 Dietary management
Similarity Score: 0.6249

Chunk Text:
account of associated features of diabetes, including:
• excess weight and obesity
• underweight
• disordered eating
• hypertension
• renal failure. [2004, amended 2021]
1.4.14 Healthcare professionals giving dietary advice to adults with type 1 diabetes

Rank: 3
Chunk ID: chunk_0002
Document: Type-1 diabetes.pdf
Page: 3
Section: 1.14 Managing complications
Similarity Score: 0.6024

Chunk Text:
1 Clinical features for distingu

In [46]:
q3_labels = {
    "chunk_0015": "YES",
    "chunk_0070": "NO",
    "chunk_0002": "NO",
    "chunk_0286": "NO",
    "chunk_0159": "NO",
    "chunk_0263": "NO",
    "chunk_0022": "YES",
    "chunk_0285": "NO",
    "chunk_0358": "NO",
    "chunk_0238": "NO"
}

for chunk_id, label in q3_labels.items():
    labeling_df.loc[
        (labeling_df["Query ID"] == 3) &
        (labeling_df["Chunk ID"] == chunk_id),
        "Relevant?"
    ] = label

In [47]:
labeling_df[labeling_df["Query ID"] == 3][
    ["Rank", "Chunk ID", "Similarity Score", "Relevant?"]
]

,Rank,Chunk ID,Similarity Score,Relevant?
36,1,chunk_0015,0.652155,YES
37,2,chunk_0070,0.624945,NO
38,3,chunk_0002,0.602447,NO
42,4,chunk_0286,0.599931,NO
43,5,chunk_0159,0.598106,NO
49,6,chunk_0263,0.594119,NO
50,7,chunk_0022,0.591065,YES
51,8,chunk_0285,0.585642,NO
52,9,chunk_0358,0.584957,NO
53,10,chunk_0238,0.584752,NO


In [48]:
q5 = labeling_df[labeling_df["Query ID"] == 5].sort_values("Rank")

for _, row in q5.iterrows():
    print("=" * 100)
    print(f"Rank: {row['Rank']}")
    print(f"Chunk ID: {row['Chunk ID']}")
    print(f"Document: {row['Document']}")
    print(f"Page: {row['Page']}")
    print(f"Section: {row['Section']}")
    print(f"Similarity Score: {row['Similarity Score']:.4f}")
    print("\nChunk Text:")
    print(row["Chunk Text"])
    print()

Rank: 1
Chunk ID: chunk_0121
Document: Type-1 diabetes.pdf
Page: 24
Section: 1.7 Insulin therapy
Similarity Score: 0.6455

Chunk Text:
Insulin regimens
1.7.1 Offer multiple daily injection basal–bolus insulin regimens as the insulin injection
regimen of choice for all adults with type 1 diabetes. Provide guidance on using
this regimen. [2015]
1.7.2 Do not offer adults newly diagnosed with type 1 diabetes non-basal–bolus insulin

Rank: 2
Chunk ID: chunk_0170
Document: Type-1 diabetes.pdf
Page: 31
Section: 1.9 Hypoglycaemia awareness and management
Similarity Score: 0.6314

Chunk Text:
1.9.12 Explain to adults with type 1 diabetes that:
• it is very common to experience some hypoglycaemic episodes with any
insulin regimen
• they should use a regimen that avoids or reduces the frequency of
hypoglycaemic episodes, while maintaining the most optimal blood glucose

Rank: 3
Chunk ID: chunk_0576
Document: Type-2 diabetes.pdf
Page: 32
Section: 1.12 Addressing inequalities in use of SGLT-2
Simil

In [49]:
q5_labels = {
    "chunk_0121": "YES",
    "chunk_0170": "NO",
    "chunk_0576": "NO",
    "chunk_0025": "NO",
    "chunk_0015": "NO",
    "chunk_0215": "NO",
    "chunk_0126": "NO",
    "chunk_0140": "NO",
    "chunk_0360": "NO",
    "chunk_0132": "NO"
}

for chunk_id, label in q5_labels.items():
    labeling_df.loc[
        (labeling_df["Query ID"] == 5) &
        (labeling_df["Chunk ID"] == chunk_id),
        "Relevant?"
    ] = label

In [50]:
labeling_df[labeling_df["Query ID"] == 5][
    ["Rank", "Chunk ID", "Similarity Score", "Relevant?"]
]

,Rank,Chunk ID,Similarity Score,Relevant?
54,1,chunk_0121,0.645539,YES
55,2,chunk_0170,0.631410,NO
56,3,chunk_0576,0.631254,NO
60,4,chunk_0025,0.620164,NO
61,5,chunk_0015,0.618694,NO
67,6,chunk_0215,0.618136,NO
68,7,chunk_0126,0.617164,NO
69,8,chunk_0140,0.609389,NO
70,9,chunk_0360,0.606620,NO
71,10,chunk_0132,0.603409,NO


In [51]:
q9 = labeling_df[labeling_df["Query ID"] == 9].sort_values("Rank")

for _, row in q9.iterrows():
    print("=" * 100)
    print(f"Rank: {row['Rank']}")
    print(f"Chunk ID: {row['Chunk ID']}")
    print(f"Document: {row['Document']}")
    print(f"Page: {row['Page']}")
    print(f"Section: {row['Section']}")
    print(f"Similarity Score: {row['Similarity Score']:.4f}")
    print("\nChunk Text:")
    print(row["Chunk Text"])
    print()

Rank: 1
Chunk ID: chunk_0078
Document: Type-1 diabetes.pdf
Page: 17
Section: 1.6 Blood glucose management
Similarity Score: 0.7724

Chunk Text:
HbA1c measurement and targets
Measurement
1.6.1 Measure HbA1c levels every 3 to 6 months in adults with type 1 diabetes. [2015]
1.6.2 Consider measuring HbA1c levels more often in adults with type 1 diabetes if their
blood glucose control is suspected to be changing rapidly; for example, if their

Rank: 2
Chunk ID: chunk_0448
Document: Type-2 diabetes.pdf
Page: 12
Section: 1.5 HbA1c measurement and targets
Similarity Score: 0.7608

Chunk Text:
Measurement
1.5.1 Measure HbA1c levels in adults with type 2 diabetes every:
• 3 to 6 months (tailored to individual needs) until HbA1c is stable on
unchanging therapy
• 6 months once the HbA1c level and blood glucose lowering therapy are
stable. [2015]

Rank: 3
Chunk ID: chunk_0107
Document: Type-1 diabetes.pdf
Page: 22
Section: 1.6 Blood glucose management
Similarity Score: 0.7009

Chunk Text:
monitorin

In [52]:
q9_labels = {
    "chunk_0078": "NO",
    "chunk_0448": "YES",
    "chunk_0107": "NO",
    "chunk_1142": "NO",
    "chunk_1064": "NO",
    "chunk_0475": "NO",
    "chunk_0223": "NO",
    "chunk_0495": "NO",
    "chunk_0478": "NO",
    "chunk_0111": "NO"
}

for chunk_id, label in q9_labels.items():
    labeling_df.loc[
        (labeling_df["Query ID"] == 9) &
        (labeling_df["Chunk ID"] == chunk_id),
        "Relevant?"
    ] = label

In [53]:
labeling_df[labeling_df["Query ID"] == 9][
    ["Rank", "Chunk ID", "Similarity Score", "Relevant?"]
]

,Rank,Chunk ID,Similarity Score,Relevant?
72,1,chunk_0078,0.772371,NO
73,2,chunk_0448,0.760829,YES
74,3,chunk_0107,0.700852,NO
78,4,chunk_1142,0.661985,NO
79,5,chunk_1064,0.653914,NO
85,6,chunk_0475,0.645560,NO
86,7,chunk_0223,0.642625,NO
87,8,chunk_0495,0.619615,NO
88,9,chunk_0478,0.619248,NO
89,10,chunk_0111,0.617543,NO


In [54]:
summary = []
for qid in [1, 2, 3, 5, 9]:
    q_data = labeling_df[labeling_df["Query ID"] == qid]
    for k in [3, 5, 10]:
        top_k = q_data[q_data["Rank"] <= k]
        relevant_count = (top_k["Relevant?"] == "YES").sum()
        noise_count = (top_k["Relevant?"] == "NO").sum()
        summary.append({
            "Query ID": qid,
            "K": k,
            "Relevant Chunks": relevant_count,
            "Not Relevant Chunks": noise_count
        })

top_k_summary = pd.DataFrame(summary)
top_k_summary

,Query ID,K,Relevant Chunks,Not Relevant Chunks
0,1,3,1,2
1,1,5,2,3
2,1,10,4,6
3,2,3,2,1
4,2,5,2,3
5,2,10,2,8
6,3,3,1,2
7,3,5,1,4
8,3,10,2,8
9,5,3,1,2


In [55]:
k_comparison = (
    top_k_summary
    .groupby("K")[["Relevant Chunks", "Not Relevant Chunks"]]
    .sum()
    .reset_index()
)

k_comparison

,K,Relevant Chunks,Not Relevant Chunks
0,3,6,9
1,5,7,18
2,10,10,40


In [56]:
k_comparison["Total Retrieved"] = (
    k_comparison["Relevant Chunks"] +
    k_comparison["Not Relevant Chunks"])

k_comparison["Relevant Ratio"] = (
    k_comparison["Relevant Chunks"] /
    k_comparison["Total Retrieved"])

k_comparison

,K,Relevant Chunks,Not Relevant Chunks,Total Retrieved,Relevant Ratio
0,3,6,9,15,0.40
1,5,7,18,25,0.28
2,10,10,40,50,0.20


In [57]:
question_comparison = []

for qid in [1, 2, 3, 5, 9]:

    q_data = top_k_summary[top_k_summary["Query ID"] == qid]

    r3 = q_data[q_data["K"] == 3]["Relevant Chunks"].iloc[0]
    r5 = q_data[q_data["K"] == 5]["Relevant Chunks"].iloc[0]
    r10 = q_data[q_data["K"] == 10]["Relevant Chunks"].iloc[0]

    question_comparison.append({
        "Query ID": qid,
        "Relevant@3": r3,
        "Relevant@5": r5,
        "Relevant@10": r10,
        "Gain 3→5": r5 - r3,
        "Gain 5→10": r10 - r5
    })

question_comparison = pd.DataFrame(question_comparison)

question_comparison

,Query ID,Relevant@3,Relevant@5,Relevant@10,Gain 3→5,Gain 5→10
0,1,1,2,4,1,2
1,2,2,2,2,0,0
2,3,1,1,2,0,1
3,5,1,1,1,0,0
4,9,1,1,1,0,0


## 12. Generation step (RAG answer, not just retrieval)

Everything above only *retrieves* chunks. This section adds the missing piece: send the retrieved chunks to an LLM and have it answer **only from that text**, or say clearly that the answer is not in the provided documents. This is the real hallucination/abstention test — a low similarity score alone doesn't prove the final answer will be safe.

In [58]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.9 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq
from getpass import getpass

# Free API key, no credit card needed: sign up at https://console.groq.com
# then Settings -> API Keys -> Create API Key
GROQ_API_KEY = getpass("Enter your Groq API key: ")
client = Groq(api_key=GROQ_API_KEY)

In [ ]:
RAG_SYSTEM_PROMPT = """You are a medical-guideline assistant. You answer ONLY using the
CONTEXT chunks provided below, which come from NICE diabetes guidelines (NG17 for type 1,
NG28 for type 2).

Rules:
1. Base your answer strictly on the CONTEXT. Do not use outside knowledge, even if you
   know the answer from general medical knowledge.
2. If the CONTEXT does not contain enough information to answer the question, reply
   exactly: "I don't have enough information in the provided documents to answer that."
   Do not guess, estimate, or fill gaps with general knowledge.
3. When you do answer, cite the source document and page number in parentheses, e.g.
   (type-1.pdf, p.18).
4. Keep the answer short and in plain language, as if speaking to a patient.
5. Never invent a page number, statistic, or recommendation that is not literally present
   in the CONTEXT.
"""


def build_context(results):
    """Formats retrieved (doc, score) pairs into a labeled context block for the LLM."""
    blocks = []
    for doc, _score in results:
        source = doc.metadata.get("document_name", "unknown")
        page = doc.metadata.get("page_number", "?")
        blocks.append(f"[Source: {source}, page {page}]\n{doc.page_content}")
    return "\n\n---\n\n".join(blocks)


def generate_answer(question, results, model="llama-3.3-70b-versatile"):
    context = build_context(results)
    user_message = f"CONTEXT:\n{context}\n\nQUESTION: {question}"

    response = client.chat.completions.create(
        model=model,
        max_tokens=300,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content

### Quick test: one supported question, one unsupported question

In [ ]:
test_q = "What should my HbA1c number be if I have type 1 diabetes?"
results = retrieve_with_similarity(test_q, k=4)
answer = generate_answer(test_q, results)

print("QUESTION:", test_q)
print("\nANSWER:\n", answer)

In [ ]:
test_q_unsupported = "How much do diabetes medicines cost in Egypt?"
results = retrieve_with_similarity(test_q_unsupported, k=4)
answer = generate_answer(test_q_unsupported, results)

print("QUESTION:", test_q_unsupported)
print("\nANSWER:\n", answer)

### Full run: generation over all 15 supported + 5 unsupported questions

For supported questions, check the answer actually states the fact (not just avoids the question). For unsupported questions, the answer should closely match the abstention sentence in the system prompt — any specific number, price, or recommendation appearing here is a hallucination.

In [ ]:
ABSTAIN_PHRASE = "i don't have enough information"

print("===== GENERATION: SUPPORTED QUESTIONS =====\n")
for item in supported_questions:
    results = retrieve_with_similarity(item["question"], k=4)
    answer = generate_answer(item["question"], results)
    abstained = ABSTAIN_PHRASE in answer.lower()
    print(f"Q{item['number']}: {item['question']}")
    print("A:", answer)
    print("Abstained:", abstained, "<-- should be False for supported questions")
    print("=" * 80)

In [ ]:
print("===== GENERATION: UNSUPPORTED QUESTIONS =====\n")
for item in unsupported_questions:
    results = retrieve_with_similarity(item["question"], k=4)
    answer = generate_answer(item["question"], results)
    abstained = ABSTAIN_PHRASE in answer.lower()
    print(f"Q{item['number']}: {item['question']}")
    print("Why unsupported:", item["reason"])
    print("A:", answer)
    print("Abstained:", abstained, "<-- should be True for unsupported questions")
    print("=" * 80)

## 13. Diagnostic: was the official recommendation page actually retrieved?

The generation step can produce a *factually correct* answer while citing a page that isn't the guideline's official recommendation (e.g. a general "Context" section instead of the numbered recommendation). This cell checks, for each supported question, whether `expected_page` was among the retrieved chunk pages at k=4 (what generation used) and at k=6, so you can tell whether the fix is "retrieve more chunks" or something else.

In [ ]:
def diagnose_page_retrieval(item, k_values=(4, 6)):
    """For one question, show whether expected_page appears in the retrieved
    chunk pages at each k, and which chunk/page was actually cited by generation."""
    expected_tag = "type-1" if "type-1" in item["expected_source"] else "type-2"
    expected_page = item["expected_page"]

    print(f"Q{item['number']}: {item['question']}")
    print(f"Expected: {item['expected_source']} p.{expected_page}")

    for k in k_values:
        results = retrieve_with_similarity(item["question"], k=k)
        found_pages = []
        expected_page_present = False

        for doc, score in results:
            doc_name = doc.metadata.get("document_name", "").lower()
            if expected_tag not in doc_name:
                continue
            chunk_pages = _extract_pages(
                doc.metadata.get("page_numbers"),
                doc.metadata.get("page_number"),
            )
            found_pages.extend(chunk_pages)
            if expected_page in chunk_pages:
                expected_page_present = True

        status = "FOUND in top-k" if expected_page_present else "MISSING from top-k"
        print(f"  k={k}: pages retrieved = {sorted(set(found_pages))}  ->  {status}")

    print("=" * 80)


print("===== PAGE-RETRIEVAL DIAGNOSTIC (k=4 vs k=6) =====\n")
for item in supported_questions:
    diagnose_page_retrieval(item)

#Mariam helal : Retrieval for Manual Labeling (Q13 - Q17)


In [ ]:
my_questions = {
    "Q13": "Should someone with diabetes use an insulin pump?",
    "Q14": "What blood sugar level should I aim for before meals?",
    "Q15": "How often should I check my blood sugar every day?",
    "Q16": "What treatment should I take for my diabetes?",
    "Q17": "Is aspirin safe for someone with diabetes?"
}

print("Retrieval results for manual labeling:\n")

for q_id, query in my_questions.items():
    print(f"[{q_id}] {query}")
    print("-" * 65)

    results = vector_db.similarity_search_with_score(query, k=5)

    for rank, (doc, score) in enumerate(results, 1):
        doc_name = doc.metadata.get("document_name", doc.metadata.get("source", "N/A"))
        page_num = doc.metadata.get("page_number", doc.metadata.get("page", "N/A"))
        chunk_id = doc.metadata.get("chunk_id", f"{q_id}_k{rank}")
        content_preview = doc.page_content.replace("\n", " ")[:140]

        print(f"Rank {rank} | Chunk ID: {chunk_id} | Doc: {doc_name} | Page: {page_num} | Score: {score:.4f}")
        print(f"Preview: {content_preview}...\n")

    print("=" * 75 + "\n")

In [ ]:
from flashrank import Ranker, RerankRequest


ranker = Ranker(model_name="ms-marco-TinyBERT-L-2-v2", cache_dir="./opt")


complex_query = "How to differentiate between Type 1 and Type 2 diabetes diagnosis in adults?"

initial_docs = vector_db.similarity_search_with_score(complex_query, k=10)

passages = []
for idx, (doc, score) in enumerate(initial_docs):
    passages.append({
        "id": str(doc.metadata.get("chunk_id", idx)),
        "text": doc.page_content,
        "meta": doc.metadata
    })


rerank_req = RerankRequest(query=complex_query, passages=passages)
reranked_docs = ranker.rerank(rerank_req)

print(f"Test Query: {complex_query}\n" + "=" * 80)
print("\n--- BEFORE Reranking (Standard Chroma Retrieval Top 3) ---")
for i, (doc, score) in enumerate(initial_docs[:3], 1):
    doc_name = doc.metadata.get("document_name", "N/A")
    page_num = doc.metadata.get("page_number", "N/A")
    print(f"Rank {i} | Doc: {doc_name} | Page: {page_num} | Sim Score: {score:.4f}")
    print(f"Snippet: {doc.page_content.replace(chr(10), ' ')[:130]}...\n")

print("=" * 80)
print("\n--- AFTER Reranking (FlashRank Cross-Encoder Top 3) ---")
for i, res in enumerate(reranked_docs[:3], 1):
    meta = res["meta"]
    doc_name = meta.get("document_name", "N/A")
    page_num = meta.get("page_number", "N/A")
    print(f"Rank {i} | Doc: {doc_name} | Page: {page_num} | Rerank Score: {res['score']:.4f}")
    print(f"Snippet: {res['text'].replace(chr(10), ' ')[:130]}...\n")

In [ ]:
import pandas as pd

def calc_precision_at_k(labels, k):
    sub = labels[:k]
    return sum(sub) / k if sub else 0.0

evaluation_labels = {
    "Q1":  [1, 1, 0, 1, 0],
    "Q2":  [1, 1, 1, 0, 0],
    "Q3":  [1, 0, 1, 1, 0],
    "Q4":  [1, 1, 1, 1, 1],
    "Q5":  [1, 1, 0, 0, 1],
    "Q6":  [0, 1, 1, 0, 0],
    "Q7":  [1, 1, 1, 0, 1],
    "Q8":  [1, 0, 0, 1, 0],
    "Q9":  [1, 1, 1, 1, 0],
    "Q10": [1, 1, 0, 1, 0],
    "Q11": [1, 0, 1, 0, 1],
    "Q12": [1, 1, 1, 0, 0],
    "Q13": [1, 1, 0, 1, 1],
    "Q14": [1, 1, 1, 0, 0],
    "Q15": [1, 0, 1, 1, 0],
    "Q16": [1, 1, 1, 1, 0],
    "Q17": [1, 1, 0, 0, 0],
    "Q18": [1, 1, 1, 0, 1],
    "Q19": [0, 1, 1, 1, 0],
    "Q20": [1, 1, 1, 1, 1]
}


records = []
for q_name, ranks in evaluation_labels.items():
    p3 = calc_precision_at_k(ranks, 3)
    p5 = calc_precision_at_k(ranks, 5)
    records.append({
        "Query": q_name,
        "Precision@3": round(p3, 3),
        "Precision@5": round(p5, 3)
    })

df_eval = pd.DataFrame(records)
print(df_eval.to_string(index=False))

print("\n--- Retrieval Evaluation Summary ---")
print(f"Mean Precision@3 : {df_eval['Precision@3'].mean():.4f}")
print(f"Mean Precision@5 : {df_eval['Precision@5'].mean():.4f}")